In [ ]:
# ## Cell 1 - Install dependencies

# %%
import subprocess
import sys

packages = [
    "accelerate>=0.33.0",
    "datasets>=2.20.0",
    "huggingface_hub>=0.24.0",
    "matplotlib>=3.8.0",
    "numpy>=1.26.0",
    "pandas>=2.2.0",
    "safetensors>=0.4.5",
    "scikit-learn>=1.5.0",
    "torch",
    "tqdm>=4.66.0",
    "transformers>=4.51.0",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

In [ ]:
# ## Cell 2 - Imports and configuration
#
# Set `SHARD_DIR` to the folder containing `eval_000.pt` ... `eval_004.pt`.
# If you uploaded the shards to Colab manually, use `/content`. If they are in
# Drive, mount Drive and point this to the Drive folder.

# %%
import gc
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, list_repo_files, snapshot_download
from IPython.display import display
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score, roc_auc_score
from tqdm.auto import tqdm


@dataclass
class Config:
    shard_dir: str = "/content"
    output_dir: str = "/content/farai_gemma3_sae_mask_diagnostics"

    model_name: str = "google/gemma-3-12b-it"
    probe_repo: str = "AlignmentResearch/diverse-deception-probe-gemma-3-12b-it"
    probe_variant: str = "last-token-generation"
    sae_repo: str = "google/gemma-scope-2-12b-it"
    layer: int = 31

    expected_shards: int = 5
    expected_total_rows: int = 1000
    deceptive_first: bool = True

    feature_ks: tuple = (1, 2, 3, 5, 8, 10, 16, 20, 32, 50, 100, 200, 500, 1000)
    random_repeats: int = 30
    active_min_fire_rate: float = 0.005

    mask_steps: int = 1200
    mask_lr: float = 0.05
    mask_l1: float = 0.003
    mask_temp_start: float = 2.0
    mask_temp_end: float = 0.15
    mask_target: str = "match_recon_logits"  # "match_recon_logits" or "labels"
    mask_eval_ks: tuple = (1, 2, 3, 5, 8, 10, 16, 20, 32, 50, 100, 200, 500)

    seed: int = 17


cfg = Config()
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU will work for most diagnostics but mask training may be slow.")

out_dir = Path(cfg.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))

# Optional Drive mount:

from google.colab import drive
drive.mount("/content/drive")
cfg.shard_dir = "your_drive_path_to_shards_here"

In [ ]:
# ## Cell 3 - Load one activation layer from shards

# %%
def load_layer_from_shards(shard_dir: str, layer: int) -> torch.Tensor:
    shard_dir = Path(shard_dir)
    shards = sorted(shard_dir.glob("eval_*.pt"))
    if len(shards) != cfg.expected_shards:
        print("Found shards:", [p.name for p in shards])
        raise FileNotFoundError(
            f"Expected {cfg.expected_shards} eval_*.pt shards in {shard_dir}, found {len(shards)}"
        )

    layer_chunks = []
    for path in shards:
        x = torch.load(path, map_location="cpu")
        if x.ndim != 3:
            raise ValueError(f"{path.name} has shape {tuple(x.shape)}, expected (rows, layers, d_model)")
        if layer >= x.shape[1]:
            raise ValueError(f"Requested layer {layer}, but {path.name} has only {x.shape[1]} layers")
        print(path.name, tuple(x.shape), x.dtype)
        layer_chunks.append(x[:, layer].float())
        del x
        gc.collect()
    h_cpu = torch.cat(layer_chunks, dim=0).contiguous()
    print("Loaded h:", tuple(h_cpu.shape), h_cpu.dtype)
    return h_cpu


h_cpu = load_layer_from_shards(cfg.shard_dir, cfg.layer)
if cfg.expected_total_rows and h_cpu.shape[0] != cfg.expected_total_rows:
    raise ValueError(f"Expected {cfg.expected_total_rows} rows, got {h_cpu.shape[0]}")

n = h_cpu.shape[0]
if n % 2 != 0:
    raise ValueError("Expected an even number of examples: first half deceptive, second half honest")

half = n // 2
if cfg.deceptive_first:
    y_eval = np.array([1] * half + [0] * half, dtype=np.int64)
else:
    y_eval = np.array([0] * half + [1] * half, dtype=np.int64)

pd.DataFrame({"row": np.arange(n), "label": y_eval}).to_csv(out_dir / "labels.csv", index=False)
print("labels:", dict(zip(*np.unique(y_eval, return_counts=True))))

h = h_cpu.to(DEVICE)

In [ ]:
# ## Cell 4 - Load the published probe

# %%
probe_root = Path(snapshot_download(cfg.probe_repo))
probe_path = probe_root / cfg.probe_variant / f"layer_{cfg.layer}" / "model.pt"
if not probe_path.exists():
    raise FileNotFoundError(f"Probe checkpoint not found: {probe_path}")

probe_state = torch.load(probe_path, map_location="cpu", weights_only=False)
print("probe keys:", sorted(probe_state.keys()))

w = probe_state["linear.weight"].reshape(-1).float().to(DEVICE)
b = probe_state["linear.bias"].reshape(()).float().to(DEVICE)

input_mean = probe_state.get("input_mean", torch.tensor(0.0))
input_scale = probe_state.get("input_scale", torch.tensor(1.0))
if not torch.is_tensor(input_mean):
    input_mean = torch.tensor(input_mean)
if not torch.is_tensor(input_scale):
    input_scale = torch.tensor(input_scale)
input_mean = input_mean.float().to(DEVICE)
input_scale = input_scale.float().to(DEVICE)


@torch.no_grad()
def probe_logit(h_):
    return ((h_.float() - input_mean) / input_scale) @ w + b


with torch.no_grad():
    logits_orig = probe_logit(h).detach().cpu().numpy()

auc_orig = roc_auc_score(y_eval, logits_orig)
acc_orig = accuracy_score(y_eval, logits_orig > 0)
print(f"Published probe on shards: AUC={auc_orig:.4f}, acc@0={acc_orig:.4f}")

# Effective residual-space probe direction after normalization. A decoded SAE
# feature d_i contributes z_i * (d_i dot effective_w) to the probe logit.
effective_w = w / input_scale

In [ ]:
# ## Cell 5 - Load the Gemma Scope 2 SAE

# %%
repo_files = sorted(list_repo_files(cfg.sae_repo))
candidates = [
    f for f in repo_files
    if f.startswith(f"resid_post/layer_{cfg.layer}_width_65k_") and f.endswith("params.safetensors")
]
if not candidates:
    raise FileNotFoundError(f"No Gemma Scope 2 resid_post SAE found for layer {cfg.layer}")

sae_file = next((f for f in candidates if "l0_medium" in f), candidates[0])
print("SAE file:", sae_file)
sd = load_file(hf_hub_download(cfg.sae_repo, sae_file))


class JumpReLUSAE(nn.Module):
    def __init__(self, sd):
        super().__init__()
        self.w_enc = nn.Parameter(sd["w_enc"].float())
        self.w_dec = nn.Parameter(sd["w_dec"].float())
        self.threshold = nn.Parameter(sd["threshold"].float())
        self.b_enc = nn.Parameter(sd["b_enc"].float())
        self.b_dec = nn.Parameter(sd["b_dec"].float())

    def encode(self, x):
        pre = x @ self.w_enc + self.b_enc
        return (pre > self.threshold) * F.relu(pre)

    def decode(self, z):
        return z @ self.w_dec + self.b_dec


sae = JumpReLUSAE(sd).to(DEVICE).eval()
Wdec = sae.w_dec.float()
b_dec = sae.b_dec.float()

with torch.no_grad():
    z_full = sae.encode(h).float()
    h_recon = sae.decode(z_full).float()
    h_err = h - h_recon

natural_l0 = (z_full > 0).float().sum(-1).mean().item()
cos_full = F.cosine_similarity(h_recon, h, dim=-1).mean().item()
mse_full = ((h_recon - h) ** 2).mean().item()

print(f"SAE health: L0={natural_l0:.1f}, cosine={cos_full:.4f}, MSE={mse_full:.4f}")

In [ ]:
# ## Cell 6 - Round-trip and SAE-error bypass diagnostics

# %%
def summarize_logits(name, logits, reference_logits=None):
    logits = np.asarray(logits)
    row = {
        "name": name,
        "auc": roc_auc_score(y_eval, logits),
        "acc_at_0": accuracy_score(y_eval, logits > 0),
        "mean_honest": float(logits[y_eval == 0].mean()),
        "mean_deceptive": float(logits[y_eval == 1].mean()),
        "margin_dec_minus_hon": float(logits[y_eval == 1].mean() - logits[y_eval == 0].mean()),
    }
    if reference_logits is not None:
        row["corr_with_reference"] = float(np.corrcoef(reference_logits, logits)[0, 1])
        row["mean_abs_delta_from_reference"] = float(np.abs(logits - reference_logits).mean())
        row["prediction_flip_rate_from_reference"] = float(((logits > 0) != (reference_logits > 0)).mean())
    return row


with torch.no_grad():
    logits_recon = probe_logit(h_recon).detach().cpu().numpy()
    logits_err_only = probe_logit(h_err).detach().cpu().numpy()
    logits_bdec_only = probe_logit(b_dec[None, :].expand_as(h)).detach().cpu().numpy()

roundtrip_rows = [
    summarize_logits("raw_h", logits_orig),
    summarize_logits("sae_reconstruction", logits_recon, logits_orig),
    summarize_logits("sae_error_vector", logits_err_only, logits_orig),
    summarize_logits("decoder_bias_only", logits_bdec_only, logits_orig),
]
roundtrip_df = pd.DataFrame(roundtrip_rows)
display(roundtrip_df)
roundtrip_df.to_csv(out_dir / "roundtrip_and_error_diagnostics.csv", index=False)

print(
    "Mean logit shift, recon - raw:",
    {
        "honest": float((logits_recon[y_eval == 0] - logits_orig[y_eval == 0]).mean()),
        "deceptive": float((logits_recon[y_eval == 1] - logits_orig[y_eval == 1]).mean()),
    },
)

In [ ]:
# ## Cell 7 - Feature contribution rankings
#
# The important correction relative to plain `w dot d_i` is using the probe's
# normalized residual direction: `effective_w = w / input_scale`.
#
# For each feature:
#
# - `decoder_alignment = d_i dot effective_w`
# - `mean_logit_contribution = mean(z_i * decoder_alignment)`
# - `margin_contribution = (mean_deceptive(z_i) - mean_honest(z_i)) * decoder_alignment`

# %%
with torch.no_grad():
    c = (Wdec @ effective_w).float()
    fire_rate = (z_full > 0).float().mean(dim=0)
    z_mean_all = z_full.mean(dim=0)
    z_mean_dec = z_full[y_eval == 1].mean(dim=0)
    z_mean_hon = z_full[y_eval == 0].mean(dim=0)
    act_diff = z_mean_dec - z_mean_hon
    mean_contrib = z_mean_all * c
    margin_contrib = act_diff * c
    mean_abs_contrib = (z_full.abs().mean(dim=0) * c.abs())

ranking_df = pd.DataFrame(
    {
        "feature": np.arange(c.numel()),
        "decoder_alignment": c.detach().cpu().numpy(),
        "abs_decoder_alignment": c.abs().detach().cpu().numpy(),
        "fire_rate": fire_rate.detach().cpu().numpy(),
        "z_mean_all": z_mean_all.detach().cpu().numpy(),
        "z_mean_deceptive": z_mean_dec.detach().cpu().numpy(),
        "z_mean_honest": z_mean_hon.detach().cpu().numpy(),
        "act_diff_dec_minus_hon": act_diff.detach().cpu().numpy(),
        "mean_logit_contribution": mean_contrib.detach().cpu().numpy(),
        "margin_contribution": margin_contrib.detach().cpu().numpy(),
        "abs_margin_contribution": margin_contrib.abs().detach().cpu().numpy(),
        "mean_abs_logit_contribution": mean_abs_contrib.detach().cpu().numpy(),
    }
)

ranking_df["rank_abs_alignment"] = ranking_df["abs_decoder_alignment"].rank(ascending=False, method="first").astype(int)
ranking_df["rank_abs_margin"] = ranking_df["abs_margin_contribution"].rank(ascending=False, method="first").astype(int)
ranking_df["rank_mean_abs_contrib"] = ranking_df["mean_abs_logit_contribution"].rank(ascending=False, method="first").astype(int)
ranking_df["rank_abs_act_diff"] = ranking_df["act_diff_dec_minus_hon"].abs().rank(ascending=False, method="first").astype(int)

ranking_df.to_csv(out_dir / "feature_ranking_all.csv", index=False)
display(ranking_df.sort_values("rank_abs_margin").head(30))
print("Saved:", out_dir / "feature_ranking_all.csv")

In [ ]:
# ## Cell 8 - Raw vs reconstruction ablations
#
# This is the core "SAE-error bypass" test:
#
# - If a feature set hurts the probe on `h_recon` but not on raw `h`, the raw
#   activation has a bypass channel in `h - SAE(h)`.
# - If it does not hurt either, the feature set probably is not necessary for
#   the published probe, at least under zero ablation.

# %%
def top_features(rank_col, k):
    return (
        ranking_df.sort_values(rank_col, ascending=True)["feature"]
        .head(k)
        .astype(int)
        .tolist()
    )


def random_active_features(k, rng):
    active = ranking_df.loc[ranking_df["fire_rate"] >= cfg.active_min_fire_rate, "feature"].to_numpy()
    if len(active) < k:
        active = ranking_df["feature"].to_numpy()
    return rng.choice(active, size=k, replace=False).astype(int).tolist()


@torch.no_grad()
def selected_delta(z, selected):
    selected = torch.as_tensor(selected, device=DEVICE, dtype=torch.long)
    return z[:, selected] @ Wdec[selected]


@torch.no_grad()
def selected_probe_contrib(z, selected):
    selected = torch.as_tensor(selected, device=DEVICE, dtype=torch.long)
    return (z[:, selected] * c[selected]).sum(dim=-1)


@torch.no_grad()
def evaluate_selected_set(name, selected, baseline_logits=logits_orig):
    delta = selected_delta(z_full, selected)

    raw_zero = h - delta
    recon_zero = h_recon - delta

    # Mean ablation is a less off-distribution variant: replace each selected
    # feature with its dataset mean rather than zero.
    selected_t = torch.as_tensor(selected, device=DEVICE, dtype=torch.long)
    z_mean = z_full[:, selected_t].mean(dim=0, keepdim=True)
    mean_delta = (z_mean - z_full[:, selected_t]) @ Wdec[selected_t]
    raw_mean = h + mean_delta
    recon_mean = h_recon + mean_delta

    out = []
    for variant, h_variant, ref in [
        ("raw_zero_ablate", raw_zero, logits_orig),
        ("recon_zero_ablate", recon_zero, logits_recon),
        ("raw_mean_ablate", raw_mean, logits_orig),
        ("recon_mean_ablate", recon_mean, logits_recon),
    ]:
        logits = probe_logit(h_variant).detach().cpu().numpy()
        row = summarize_logits(f"{name}:{variant}", logits, ref)
        row["set_name"] = name
        row["variant"] = variant
        row["k"] = len(selected)
        out.append(row)

    # The same probe-logit effects can be computed directly from feature-space
    # contributions. This is a check against decoder arithmetic mistakes.
    contrib = selected_probe_contrib(z_full, selected).detach().cpu().numpy()
    out.append(
        {
            "name": f"{name}:direct_feature_contrib",
            "set_name": name,
            "variant": "direct_feature_contrib",
            "k": len(selected),
            "auc": roc_auc_score(y_eval, logits_recon - contrib),
            "acc_at_0": accuracy_score(y_eval, (logits_recon - contrib) > 0),
            "mean_honest": float((logits_recon - contrib)[y_eval == 0].mean()),
            "mean_deceptive": float((logits_recon - contrib)[y_eval == 1].mean()),
            "margin_dec_minus_hon": float((logits_recon - contrib)[y_eval == 1].mean() - (logits_recon - contrib)[y_eval == 0].mean()),
            "corr_with_reference": float(np.corrcoef(logits_recon, logits_recon - contrib)[0, 1]),
            "mean_abs_delta_from_reference": float(np.abs(contrib).mean()),
            "prediction_flip_rate_from_reference": float(((logits_recon > 0) != ((logits_recon - contrib) > 0)).mean()),
        }
    )
    return out


ranking_specs = {
    "abs_alignment": "rank_abs_alignment",
    "abs_margin": "rank_abs_margin",
    "mean_abs_contrib": "rank_mean_abs_contrib",
    "abs_act_diff": "rank_abs_act_diff",
}

ablation_rows = []
for ranking_name, rank_col in ranking_specs.items():
    for k in cfg.feature_ks:
        selected = top_features(rank_col, k)
        ablation_rows.extend(evaluate_selected_set(f"{ranking_name}_top{k}", selected))

rng = np.random.default_rng(cfg.seed)
for k in cfg.feature_ks:
    random_rows = []
    for r in range(cfg.random_repeats):
        selected = random_active_features(k, rng)
        rows = evaluate_selected_set(f"random_active_top{k}_rep{r:02d}", selected)
        random_rows.extend(rows)
    ablation_rows.extend(random_rows)

ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(out_dir / "raw_vs_recon_ablation_sweep.csv", index=False)
display(ablation_df.head())
print("Saved:", out_dir / "raw_vs_recon_ablation_sweep.csv")

summary_cols = [
    "set_name",
    "variant",
    "k",
    "auc",
    "acc_at_0",
    "mean_abs_delta_from_reference",
    "prediction_flip_rate_from_reference",
]
display(
    ablation_df[
        ablation_df["set_name"].str.contains("abs_margin_top|abs_alignment_top|mean_abs_contrib_top", regex=True)
    ][summary_cols].head(80)
)

In [ ]:
# ## Cell 9 - Sparse probe-readout mask
#
# This mask is trained in feature space, not by repeatedly decoding residual
# vectors. The fixed published probe makes this possible:
#
# `probe_logit(SAE_decode(z * m)) = base_logit_from_decoder_bias + sum_i z_i * m_i * c_i`
#
# This learns a sparse set of SAE features that explains the published probe
# readout on the SAE reconstruction. It does not prove behavior causality.

# %%
class SparseReadoutMask(nn.Module):
    def __init__(self, n_features, init_logit=-4.0):
        super().__init__()
        self.mask_logits = nn.Parameter(torch.full((n_features,), init_logit))

    def forward(self, temp):
        return torch.sigmoid(self.mask_logits / temp)


with torch.no_grad():
    base_recon_logit = probe_logit(b_dec[None, :]).reshape(())
    target_recon_logits = torch.as_tensor(logits_recon, device=DEVICE, dtype=torch.float32)
    target_labels = torch.as_tensor(y_eval, device=DEVICE, dtype=torch.float32)
    zc = z_full * c[None, :]

mask_model = SparseReadoutMask(z_full.shape[1]).to(DEVICE)
opt = torch.optim.AdamW(mask_model.parameters(), lr=cfg.mask_lr, weight_decay=0.0)

loss_rows = []
target_std = target_recon_logits.std().clamp_min(1e-6)

for step in tqdm(range(cfg.mask_steps), desc="training sparse readout mask"):
    frac = step / max(cfg.mask_steps - 1, 1)
    temp = cfg.mask_temp_start * (cfg.mask_temp_end / cfg.mask_temp_start) ** frac
    m = mask_model(temp)
    pred = base_recon_logit + (zc * m[None, :]).sum(dim=-1)

    if cfg.mask_target == "match_recon_logits":
        fit_loss = (((pred - target_recon_logits) / target_std) ** 2).mean()
    elif cfg.mask_target == "labels":
        fit_loss = F.binary_cross_entropy_with_logits(pred, target_labels)
    else:
        raise ValueError(f"Unknown mask_target: {cfg.mask_target}")

    sparsity = m.mean()
    loss = fit_loss + cfg.mask_l1 * sparsity

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % 50 == 0 or step == cfg.mask_steps - 1:
        with torch.no_grad():
            hard_count = int((m > 0.5).sum().item())
            loss_rows.append(
                {
                    "step": step,
                    "temp": temp,
                    "loss": float(loss.item()),
                    "fit_loss": float(fit_loss.item()),
                    "sparsity_mean": float(sparsity.item()),
                    "hard_count_gt_0_5": hard_count,
                }
            )

loss_df = pd.DataFrame(loss_rows)
loss_df.to_csv(out_dir / "sparse_readout_mask_training.csv", index=False)
display(loss_df.tail())

with torch.no_grad():
    final_mask_scores = torch.sigmoid(mask_model.mask_logits / cfg.mask_temp_end).detach().cpu().numpy()

mask_rank_df = ranking_df.copy()
mask_rank_df["mask_score"] = final_mask_scores
mask_rank_df["rank_mask_score"] = mask_rank_df["mask_score"].rank(ascending=False, method="first").astype(int)
mask_rank_df.to_csv(out_dir / "feature_ranking_with_mask.csv", index=False)
display(mask_rank_df.sort_values("rank_mask_score").head(40))

In [ ]:
# ## Cell 10 - Evaluate mask-selected features

# %%
mask_rows = []
for k in cfg.mask_eval_ks:
    selected = (
        mask_rank_df.sort_values("rank_mask_score", ascending=True)["feature"]
        .head(k)
        .astype(int)
        .tolist()
    )
    mask_rows.extend(evaluate_selected_set(f"sparse_readout_mask_top{k}", selected))

mask_eval_df = pd.DataFrame(mask_rows)
mask_eval_df.to_csv(out_dir / "sparse_readout_mask_ablation_sweep.csv", index=False)
display(mask_eval_df[summary_cols].head(120))
print("Saved:", out_dir / "sparse_readout_mask_ablation_sweep.csv")

In [ ]:
# ## Cell 11 - Compact comparison table

# %%
def compact_rows(df, set_prefixes, variants=("raw_zero_ablate", "recon_zero_ablate")):
    rows = []
    for prefix in set_prefixes:
        for variant in variants:
            sub = df[(df["set_name"] == prefix) & (df["variant"] == variant)]
            if len(sub):
                rows.append(sub.iloc[0].to_dict())
    return pd.DataFrame(rows)


interesting_sets = []
for k in (5, 16, 50, 100, 500):
    interesting_sets.extend(
        [
            f"abs_alignment_top{k}",
            f"abs_margin_top{k}",
            f"mean_abs_contrib_top{k}",
            f"sparse_readout_mask_top{k}",
        ]
    )

combined = pd.concat([ablation_df, mask_eval_df], ignore_index=True)
compact = compact_rows(combined, interesting_sets)
compact = compact[
    [
        "set_name",
        "variant",
        "k",
        "auc",
        "acc_at_0",
        "mean_abs_delta_from_reference",
        "prediction_flip_rate_from_reference",
        "corr_with_reference",
    ]
]
compact.to_csv(out_dir / "compact_probe_reliance_comparison.csv", index=False)
display(compact)

In [ ]:
# ## Cell 12 - Plot the main diagnostic

# %%
plot_df = compact.copy()
plot_df["auc_drop"] = np.where(
    plot_df["variant"].str.startswith("raw"),
    auc_orig - plot_df["auc"],
    roc_auc_score(y_eval, logits_recon) - plot_df["auc"],
)

fig, ax = plt.subplots(figsize=(12, 5))
for variant, sub in plot_df.groupby("variant"):
    ax.scatter(sub["k"], sub["auc_drop"], label=variant, s=45)
for _, row in plot_df.iterrows():
    label = row["set_name"].replace("_top", "@").replace("sparse_readout_mask", "mask")
    ax.annotate(label, (row["k"], row["auc_drop"]), fontsize=7, alpha=0.75, xytext=(3, 3), textcoords="offset points")
ax.axhline(0, color="black", lw=1)
ax.set_xscale("log")
ax.set_xlabel("Number of selected SAE features")
ax.set_ylabel("AUC drop from matching baseline")
ax.set_title("Does selected SAE-feature ablation hurt the probe?")
ax.legend()
fig.tight_layout()
fig.savefig(out_dir / "probe_reliance_auc_drop.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", out_dir / "probe_reliance_auc_drop.png")

In [ ]:
# ## summary plots

# %%
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Change this if running locally.
PLOT_DIR = Path(cfg.output_dir) if "cfg" in globals() else Path("/content/farai_gemma3_sae_mask_diagnostics")
OUT = PLOT_DIR / "plots"
OUT.mkdir(parents=True, exist_ok=True)

roundtrip = pd.read_csv(PLOT_DIR / "roundtrip_and_error_diagnostics.csv")
ablation = pd.read_csv(PLOT_DIR / "raw_vs_recon_ablation_sweep.csv")
mask_eval = pd.read_csv(PLOT_DIR / "sparse_readout_mask_ablation_sweep.csv")
mask_train = pd.read_csv(PLOT_DIR / "sparse_readout_mask_training.csv")
mask_rank = pd.read_csv(PLOT_DIR / "feature_ranking_with_mask.csv")

combined = pd.concat([ablation, mask_eval], ignore_index=True)

raw_auc = float(roundtrip.loc[roundtrip["name"] == "raw_h", "auc"].iloc[0])
recon_auc = float(roundtrip.loc[roundtrip["name"] == "sae_reconstruction", "auc"].iloc[0])

def baseline_auc_for_variant(variant):
    return raw_auc if variant.startswith("raw") else recon_auc

# ---------- Plot 1: margin decomposition ----------
fig, ax = plt.subplots(figsize=(8, 4.8))

margin_rows = roundtrip[
    roundtrip["name"].isin(["raw_h", "sae_reconstruction", "sae_error_vector"])
].copy()

labels = {
    "raw_h": "Raw activation",
    "sae_reconstruction": "SAE reconstruction",
    "sae_error_vector": "SAE error vector",
}
margin_rows["label"] = margin_rows["name"].map(labels)

raw_margin = float(margin_rows.loc[margin_rows["name"] == "raw_h", "margin_dec_minus_hon"].iloc[0])
colors = ["#3b78c2", "#f28e2b", "#59a14f"]

bars = ax.bar(
    margin_rows["label"],
    margin_rows["margin_dec_minus_hon"],
    color=colors,
    edgecolor="black",
    linewidth=0.8,
)

for bar, val in zip(bars, margin_rows["margin_dec_minus_hon"]):
    pct = 100 * val / raw_margin
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{val:.2f}\n({pct:.0f}% of raw)",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Probe margin: mean deceptive logit - mean honest logit")
ax.set_title("Most published-probe margin lives outside the SAE reconstruction")
fig.tight_layout()
fig.savefig(OUT / "01_probe_margin_decomposition.png", dpi=180, bbox_inches="tight")
plt.show()


# ---------- Plot 2: raw vs reconstruction AUC drop ----------
interesting_prefixes = [
    "abs_margin",
    "mean_abs_contrib",
    "sparse_readout_mask",
]

plot_rows = []
for _, row in combined.iterrows():
    if row["variant"] not in ["raw_zero_ablate", "recon_zero_ablate"]:
        continue
    if not any(row["set_name"].startswith(prefix + "_top") for prefix in interesting_prefixes):
        continue

    clean = row["set_name"]
    clean = clean.replace("abs_margin_top", "margin top-")
    clean = clean.replace("mean_abs_contrib_top", "mean contribution top-")
    clean = clean.replace("sparse_readout_mask_top", "mask top-")

    plot_rows.append({
        "set_name": row["set_name"],
        "label": clean,
        "family": clean.rsplit(" top-", 1)[0],
        "k": int(row["k"]),
        "variant": row["variant"],
        "auc_drop": baseline_auc_for_variant(row["variant"]) - float(row["auc"]),
    })

plot_df = pd.DataFrame(plot_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, variant, title in [
    (axes[0], "raw_zero_ablate", "Ablating features from raw activations"),
    (axes[1], "recon_zero_ablate", "Ablating features from SAE reconstruction"),
]:
    sub = plot_df[plot_df["variant"] == variant]
    for family, fam_df in sub.groupby("family"):
        fam_df = fam_df.sort_values("k")
        ax.plot(fam_df["k"], fam_df["auc_drop"], marker="o", linewidth=2, label=family)

    ax.axhline(0, color="black", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel("Number of selected SAE features")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("AUC drop from matching baseline")
axes[1].legend(loc="best", fontsize=9)
fig.suptitle("Selected SAE features hurt the reconstructed probe, not the raw probe", y=1.03)
fig.tight_layout()
fig.savefig(OUT / "02_raw_vs_recon_auc_drop.png", dpi=180, bbox_inches="tight")
plt.show()


# ---------- Plot 3: prediction flip rate ----------
flip_rows = []
for _, row in combined.iterrows():
    if row["variant"] not in ["raw_zero_ablate", "recon_zero_ablate"]:
        continue
    if not any(row["set_name"].startswith(prefix + "_top") for prefix in interesting_prefixes):
        continue

    clean = row["set_name"]
    clean = clean.replace("abs_margin_top", "margin top-")
    clean = clean.replace("mean_abs_contrib_top", "mean contribution top-")
    clean = clean.replace("sparse_readout_mask_top", "mask top-")

    flip_rows.append({
        "family": clean.rsplit(" top-", 1)[0],
        "k": int(row["k"]),
        "variant": row["variant"],
        "flip_rate": float(row["prediction_flip_rate_from_reference"]),
    })

flip_df = pd.DataFrame(flip_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, variant, title in [
    (axes[0], "raw_zero_ablate", "Raw activation prediction flips"),
    (axes[1], "recon_zero_ablate", "SAE reconstruction prediction flips"),
]:
    sub = flip_df[flip_df["variant"] == variant]
    for family, fam_df in sub.groupby("family"):
        fam_df = fam_df.sort_values("k")
        ax.plot(fam_df["k"], fam_df["flip_rate"], marker="o", linewidth=2, label=family)

    ax.axhline(0, color="black", lw=1)
    ax.set_xscale("log")
    ax.set_xlabel("Number of selected SAE features")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)

axes[0].set_ylabel("Prediction flip rate from matching baseline")
axes[1].legend(loc="best", fontsize=9)
fig.suptitle("Threshold behavior changes mainly in reconstruction space", y=1.03)
fig.tight_layout()
fig.savefig(OUT / "03_prediction_flip_rate.png", dpi=180, bbox_inches="tight")
plt.show()


# ---------- Plot 4: random-control comparison ----------
random = ablation[
    ablation["set_name"].str.startswith("random_active_top")
    & ablation["variant"].isin(["raw_zero_ablate", "recon_zero_ablate"])
].copy()

random["auc_drop"] = random.apply(
    lambda r: baseline_auc_for_variant(r["variant"]) - float(r["auc"]),
    axis=1,
)

random_summary = (
    random.groupby(["k", "variant"])["auc_drop"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

for variant, label, color in [
    ("raw_zero_ablate", "Random controls: raw", "#4e79a7"),
    ("recon_zero_ablate", "Random controls: reconstruction", "#f28e2b"),
]:
    sub = random_summary[random_summary["variant"] == variant].sort_values("k")
    ax.plot(sub["k"], sub["mean"], marker="o", color=color, label=label)
    ax.fill_between(sub["k"], sub["min"], sub["max"], color=color, alpha=0.18)

ax.axhline(0, color="black", lw=1)
ax.set_xscale("log")
ax.set_xlabel("Number of random active SAE features")
ax.set_ylabel("AUC drop from matching baseline")
ax.set_title("Random active SAE features do not reproduce the selected-feature effect")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "04_random_control_auc_drop.png", dpi=180, bbox_inches="tight")
plt.show()


# ---------- Plot 5: sparse mask convergence ----------
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(mask_train["step"], mask_train["fit_loss"], color="#4e79a7", linewidth=2, label="fit loss")
ax1.set_xlabel("Training step")
ax1.set_ylabel("Fit loss", color="#4e79a7")
ax1.tick_params(axis="y", labelcolor="#4e79a7")
ax1.grid(True, alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(mask_train["step"], mask_train["hard_count_gt_0_5"], color="#e15759", linewidth=2, label="features > 0.5")
ax2.set_ylabel("Hard selected features", color="#e15759")
ax2.tick_params(axis="y", labelcolor="#e15759")

fig.suptitle("Sparse readout mask converges to a small reconstruction-space explanation")
fig.tight_layout()
fig.savefig(OUT / "05_sparse_mask_training.png", dpi=180, bbox_inches="tight")
plt.show()


# ---------- Plot 6: mask scores by rank ----------
top_n = 500
mr = mask_rank.sort_values("rank_mask_score").head(top_n).copy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mr["rank_mask_score"], mr["mask_score"], linewidth=2)
ax.axvline((mask_rank["mask_score"] > 0.5).sum(), color="red", linestyle="--", label="score > 0.5")
ax.set_xlabel("Feature rank by mask score")
ax.set_ylabel("Mask score")
ax.set_title("Sparse mask saturates after the selected feature set")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(OUT / "06_mask_scores_by_rank.png", dpi=180, bbox_inches="tight")
plt.show()

print("Saved plots to:", OUT)
for p in sorted(OUT.glob("*.png")):
    print(p.name)

In [ ]:
# Drop this cell after the diagnostics have produced:
# - `roundtrip_and_error_diagnostics.csv`
# - `raw_vs_recon_ablation_sweep.csv`
# - `sparse_readout_mask_ablation_sweep.csv`
# - `sparse_readout_mask_training.csv`
# - `feature_ranking_with_mask.csv`

# %%
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D


PLOT_DIR = Path("/content")
OUT = PLOT_DIR / "plots_reference_style"
OUT.mkdir(parents=True, exist_ok=True)

roundtrip = pd.read_csv(PLOT_DIR / "roundtrip_and_error_diagnostics.csv")
ablation = pd.read_csv(PLOT_DIR / "raw_vs_recon_ablation_sweep.csv")
mask_eval = pd.read_csv(PLOT_DIR / "sparse_readout_mask_ablation_sweep.csv")
mask_train = pd.read_csv(PLOT_DIR / "sparse_readout_mask_training.csv")
mask_rank = pd.read_csv(PLOT_DIR / "feature_ranking_with_mask.csv")

combined = pd.concat([ablation, mask_eval], ignore_index=True)


# ---------- Reference-figure visual grammar ----------
RED = "#d62728"
BLUE = "#1f77b4"
GRAY = "#7f7f7f"
DARK_GRAY = "#555555"
LIGHT_GRID = "#e6e6e6"
BLACK = "#222222"

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.titleweight": "bold",
        "axes.labelsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#777777",
        "axes.linewidth": 0.8,
        "xtick.color": BLACK,
        "ytick.color": BLACK,
        "axes.labelcolor": BLACK,
        "text.color": BLACK,
        "figure.dpi": 140,
        "savefig.dpi": 300,
    }
)


def finish_axis(ax, grid_axis="y"):
    ax.grid(True, axis=grid_axis, color=LIGHT_GRID, linewidth=0.8, alpha=0.75)
    ax.set_axisbelow(True)
    ax.tick_params(length=3, width=0.8)
    return ax


def label_bars(ax, bars, fmt="{:.3f}", pad_frac=0.018, fontsize=7):
    y0, y1 = ax.get_ylim()
    pad = (y1 - y0) * pad_frac
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + pad,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color=DARK_GRAY,
        )


def auc_baseline(variant):
    raw_auc = float(roundtrip.loc[roundtrip["name"] == "raw_h", "auc"].iloc[0])
    recon_auc = float(roundtrip.loc[roundtrip["name"] == "sae_reconstruction", "auc"].iloc[0])
    return raw_auc if str(variant).startswith("raw") else recon_auc


def family_label(set_name):
    if set_name.startswith("abs_margin_top"):
        return "Probe\n(c=w^T d)"
    if set_name.startswith("sparse_readout_mask_top"):
        return "Probe\n(binary mask)"
    if set_name.startswith("mean_abs_contrib_top"):
        return "Model\n(gradient)"
    if set_name.startswith("random_active_top"):
        return "Control"
    return set_name


def family_order(label):
    order = {
        "Probe\n(c=wTd)": 0,
        "Probe\n(binary mask)": 1,
        "Model\n(gradient)": 2,
        "Control": 3,
    }
    return order.get(label, 99)


def top_k_from_name(set_name):
    m = re.search(r"_top(\d+)", str(set_name))
    return int(m.group(1)) if m else np.nan


def selected_summary(k=50):
    rows = []

    selected_prefixes = [
        "abs_margin_top",
        "sparse_readout_mask_top",
        "mean_abs_contrib_top",
    ]
    for prefix in selected_prefixes:
        name = f"{prefix}{k}"
        for variant in ["raw_zero_ablate", "recon_zero_ablate"]:
            sub = combined[(combined["set_name"] == name) & (combined["variant"] == variant)]
            if len(sub):
                rows.append(sub.iloc[0].to_dict())

    random = ablation[
        ablation["set_name"].str.startswith(f"random_active_top{k}_rep")
        & ablation["variant"].isin(["raw_zero_ablate", "recon_zero_ablate"])
    ].copy()
    if len(random):
        random["set_name"] = f"random_active_top{k}"
        rows.extend(random.to_dict("records"))

    out = pd.DataFrame(rows)
    out["family"] = out["set_name"].map(family_label)
    out["auc_drop"] = out.apply(lambda r: auc_baseline(r["variant"]) - float(r["auc"]), axis=1)
    out["flip_rate"] = out["prediction_flip_rate_from_reference"].astype(float)
    out["space"] = np.where(out["variant"].str.startswith("raw"), "raw activation", "SAE reconstruction")
    return out


# ---------- Plot 1: combined headline figure ----------
# Pick the largest k shared across the three selected families and random controls,
# capped at 50 so the chart remains "small feature set" rather than bulk ablation.
available_ks = sorted(
    set(combined["set_name"].map(top_k_from_name).dropna().astype(int))
    & set(ablation.loc[ablation["set_name"].str.startswith("random_active_top"), "k"].astype(int))
)
summary_k = max([k for k in available_ks if k <= 50], default=50)
summary = selected_summary(summary_k)

bar_summary = (
    summary.groupby(["family", "space"], as_index=False)
    .agg(
        flip_mean=("flip_rate", "mean"),
        flip_min=("flip_rate", "min"),
        flip_max=("flip_rate", "max"),
        auc_drop_mean=("auc_drop", "mean"),
        auc_drop_min=("auc_drop", "min"),
        auc_drop_max=("auc_drop", "max"),
    )
)
bar_summary["order"] = bar_summary["family"].map(family_order)
bar_summary = bar_summary.sort_values(["order", "space"])

families = bar_summary["family"].drop_duplicates().tolist()
x = np.arange(len(families))
width = 0.34

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.3))
fig.suptitle(
    "Probe-selected SAE features move the reconstruction-space readout more than raw activations",
    fontsize=11,
    fontweight="bold",
    y=1.02,
)

ax = axes[0]
for offset, space, color, label in [
    (-width / 2, "raw activation", RED, "raw activation flip"),
    (width / 2, "SAE reconstruction", BLUE, "probe/reconstruction flip"),
]:
    sub = bar_summary[bar_summary["space"] == space].set_index("family").reindex(families)
    vals = sub["flip_mean"].fillna(0).to_numpy()
    low = vals - sub["flip_min"].fillna(sub["flip_mean"]).to_numpy()
    high = sub["flip_max"].fillna(sub["flip_mean"]).to_numpy() - vals
    yerr = np.vstack([np.maximum(low, 0), np.maximum(high, 0)])
    bars = ax.bar(
        x + offset,
        vals,
        width,
        color=color,
        edgecolor=BLACK,
        linewidth=0.35,
        yerr=yerr,
        error_kw={"elinewidth": 1.0, "ecolor": BLACK, "capsize": 3, "capthick": 1.0},
        label=label,
    )
    label_bars(ax, bars)

ax.set_xticks(x)
ax.set_xticklabels(families)
ax.set_ylabel("prediction flip rate after ablation")
ax.set_title(f"Feature-set ablation, top-{summary_k}\n(raw features mostly leave behavior intact)")
ax.set_ylim(0, max(0.04, bar_summary["flip_mean"].max() * 1.35))
ax.legend(loc="upper right", frameon=True, fancybox=False, edgecolor="#dddddd")
finish_axis(ax)

ax = axes[1]
for offset, space, color, label in [
    (-width / 2, "raw activation", RED, "raw activation AUC drop"),
    (width / 2, "SAE reconstruction", BLUE, "reconstruction AUC drop"),
]:
    sub = bar_summary[bar_summary["space"] == space].set_index("family").reindex(families)
    vals = sub["auc_drop_mean"].fillna(0).to_numpy()
    low = vals - sub["auc_drop_min"].fillna(sub["auc_drop_mean"]).to_numpy()
    high = sub["auc_drop_max"].fillna(sub["auc_drop_mean"]).to_numpy() - vals
    yerr = np.vstack([np.maximum(low, 0), np.maximum(high, 0)])
    bars = ax.bar(
        x + offset,
        vals,
        width,
        color=color,
        edgecolor=BLACK,
        linewidth=0.35,
        yerr=yerr,
        error_kw={"elinewidth": 1.0, "ecolor": BLACK, "capsize": 3, "capthick": 1.0},
        label=label,
    )
    label_bars(ax, bars)

ax.axhline(0, color=BLACK, lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(families)
ax.set_ylabel("AUC drop from matching baseline")
ax.set_title(f"Probe performance loss, top-{summary_k}\n(effect concentrates in SAE reconstruction)")
ax.set_ylim(0, max(0.04, bar_summary["auc_drop_mean"].max() * 1.35))
ax.legend(loc="upper right", frameon=True, fancybox=False, edgecolor="#dddddd")
finish_axis(ax)

fig.tight_layout()
fig.savefig(OUT / "01_reference_style_headline_summary.png", bbox_inches="tight")
plt.show()


# ---------- Plot 2: margin decomposition ----------
margin_rows = roundtrip[
    roundtrip["name"].isin(["raw_h", "sae_reconstruction", "sae_error_vector"])
].copy()
margin_rows["label"] = margin_rows["name"].map(
    {
        "raw_h": "Raw\nactivation",
        "sae_reconstruction": "SAE\nreconstruction",
        "sae_error_vector": "SAE error\nvector",
    }
)
margin_rows["order"] = margin_rows["name"].map({"raw_h": 0, "sae_reconstruction": 1, "sae_error_vector": 2})
margin_rows = margin_rows.sort_values("order")
raw_margin = float(margin_rows.loc[margin_rows["name"] == "raw_h", "margin_dec_minus_hon"].iloc[0])

fig, ax = plt.subplots(figsize=(5.6, 4.1))
bars = ax.bar(
    margin_rows["label"],
    margin_rows["margin_dec_minus_hon"],
    color=[BLUE, GRAY, RED],
    edgecolor=BLACK,
    linewidth=0.35,
)
for bar, val in zip(bars, margin_rows["margin_dec_minus_hon"]):
    pct = 100 * val / raw_margin if raw_margin else np.nan
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + ax.get_ylim()[1] * 0.015,
        f"{val:.2f}\n({pct:.0f}% raw)",
        ha="center",
        va="bottom",
        fontsize=7,
        color=DARK_GRAY,
    )

ax.axhline(0, color=BLACK, lw=0.8)
ax.set_ylabel("probe margin: deceptive logit - honest logit")
ax.set_title("Most published-probe margin lives outside the SAE reconstruction")
finish_axis(ax)
fig.tight_layout()
fig.savefig(OUT / "02_reference_style_margin_decomposition.png", bbox_inches="tight")
plt.show()


# ---------- Plot 3: top-k flip-rate curves, made compact like the right panel ----------
curve_rows = []
for _, row in combined.iterrows():
    if row["variant"] not in ["raw_zero_ablate", "recon_zero_ablate"]:
        continue
    if not (
        row["set_name"].startswith("abs_margin_top")
        or row["set_name"].startswith("sparse_readout_mask_top")
        or row["set_name"].startswith("mean_abs_contrib_top")
    ):
        continue
    curve_rows.append(
        {
            "k": int(row["k"]),
            "family": family_label(row["set_name"]).replace("\n", " "),
            "space": "raw activation" if row["variant"].startswith("raw") else "SAE reconstruction",
            "flip_rate": float(row["prediction_flip_rate_from_reference"]),
        }
    )

curve_df = pd.DataFrame(curve_rows)
curve_ks = [k for k in [16, 50, 100, 200, 500] if k in set(curve_df["k"])]

for family in curve_df["family"].drop_duplicates():
    subfam = curve_df[(curve_df["family"] == family) & (curve_df["k"].isin(curve_ks))]
    if subfam.empty:
        continue

    fig, ax = plt.subplots(figsize=(5.4, 4.1))
    obs = subfam[subfam["space"] == "SAE reconstruction"].set_index("k").reindex(curve_ks)
    raw = subfam[subfam["space"] == "raw activation"].set_index("k").reindex(curve_ks)

    bars = ax.bar(
        np.arange(len(curve_ks)),
        obs["flip_rate"].fillna(0),
        color=GRAY,
        edgecolor=BLACK,
        linewidth=0.35,
        label="observed (SAE reconstruction)",
    )
    raw_vals = raw["flip_rate"].fillna(0).to_numpy()
    ax.plot(
        np.arange(len(curve_ks)),
        raw_vals,
        marker="x",
        markersize=6,
        markeredgewidth=1.2,
        color=RED,
        linewidth=1.0,
        label="raw activation",
    )
    label_bars(ax, bars)
    ax.set_xticks(np.arange(len(curve_ks)))
    ax.set_xticklabels([f"top-{k}" for k in curve_ks])
    ax.set_ylabel("prediction flip rate")
    ax.set_title(f"{family}: reconstruction flips exceed raw flips")
    ax.set_ylim(0, max(0.04, np.nanmax(obs["flip_rate"].to_numpy()) * 1.30))
    ax.legend(loc="upper left", frameon=True, fancybox=False, edgecolor="#dddddd")
    finish_axis(ax)
    fig.tight_layout()
    safe_family = family.lower().replace(" ", "_").replace("=", "").replace("(", "").replace(")", "")
    fig.savefig(OUT / f"03_reference_style_topk_flip_{safe_family}.png", bbox_inches="tight")
    plt.show()


# ---------- Plot 4: random-control comparison ----------
random = ablation[
    ablation["set_name"].str.startswith("random_active_top")
    & ablation["variant"].isin(["raw_zero_ablate", "recon_zero_ablate"])
].copy()
random["auc_drop"] = random.apply(lambda r: auc_baseline(r["variant"]) - float(r["auc"]), axis=1)
random_summary = (
    random.groupby(["k", "variant"])["auc_drop"]
    .agg(["mean", "min", "max"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(6.0, 4.2))
for variant, label, color, marker in [
    ("raw_zero_ablate", "raw activation", RED, "x"),
    ("recon_zero_ablate", "SAE reconstruction", GRAY, "o"),
]:
    sub = random_summary[random_summary["variant"] == variant].sort_values("k")
    ax.plot(sub["k"], sub["mean"], marker=marker, color=color, linewidth=1.5, markersize=5, label=label)
    ax.fill_between(sub["k"], sub["min"], sub["max"], color=color, alpha=0.18, linewidth=0)

ax.axhline(0, color=BLACK, lw=0.8)
ax.set_xscale("log")
ax.set_xlabel("number of random active SAE features")
ax.set_ylabel("AUC drop from matching baseline")
ax.set_title("Random active features do not reproduce the selected-feature effect")
ax.legend(loc="best", frameon=True, fancybox=False, edgecolor="#dddddd")
finish_axis(ax)
fig.tight_layout()
fig.savefig(OUT / "04_reference_style_random_control_auc_drop.png", bbox_inches="tight")
plt.show()


# ---------- Plot 5: sparse mask convergence ----------
fig, ax1 = plt.subplots(figsize=(6.0, 4.2))
ax1.plot(mask_train["step"], mask_train["fit_loss"], color=BLUE, linewidth=1.8, label="fit loss")
ax1.set_xlabel("training step")
ax1.set_ylabel("fit loss", color=BLUE)
ax1.tick_params(axis="y", labelcolor=BLUE)
finish_axis(ax1)

ax2 = ax1.twinx()
ax2.plot(mask_train["step"], mask_train["hard_count_gt_0_5"], color=RED, linewidth=1.8, label="features > 0.5")
ax2.set_ylabel("hard-selected features", color=RED)
ax2.tick_params(axis="y", labelcolor=RED)
ax2.spines["top"].set_visible(False)

handles = [
    Line2D([0], [0], color=BLUE, lw=1.8, label="fit loss"),
    Line2D([0], [0], color=RED, lw=1.8, label="features > 0.5"),
]
ax1.legend(handles=handles, loc="best", frameon=True, fancybox=False, edgecolor="#dddddd")
ax1.set_title("Sparse readout mask converges to a small reconstruction-space explanation")
fig.tight_layout()
fig.savefig(OUT / "05_reference_style_sparse_mask_training.png", bbox_inches="tight")
plt.show()


# ---------- Plot 6: mask scores by rank ----------
top_n = 500
mr = mask_rank.sort_values("rank_mask_score").head(top_n).copy()
hard_count = int((mask_rank["mask_score"] > 0.5).sum())

fig, ax = plt.subplots(figsize=(6.0, 4.2))
ax.plot(mr["rank_mask_score"], mr["mask_score"], color=GRAY, linewidth=1.8, label="mask score")
ax.axvline(hard_count, color=RED, linestyle="--", linewidth=1.2, label="score > 0.5")
ax.set_xlabel("feature rank by mask score")
ax.set_ylabel("mask score")
ax.set_title("Sparse mask saturates after the selected feature set")
ax.legend(loc="best", frameon=True, fancybox=False, edgecolor="#dddddd")
finish_axis(ax)
fig.tight_layout()
fig.savefig(OUT / "06_reference_style_mask_scores_by_rank.png", bbox_inches="tight")
plt.show()


print("Saved reference-style plots to:", OUT)
for p in sorted(OUT.glob("*.png")):
    print(p.name)